# Inference Pipeline

This notebook demonstrates how to perform **inference** using the previously trained S4Casting model.  
The goal is to take recent time-series data (e.g., energy measurements) and generate short- or medium-term forecasts.

Workflow:
1. Load the configuration the model was trained with
2. Prepare an input DataFrame.
3. Run the model checkpoint to predict the next horizon.
4. Inspect, plot, and optionally persist results.

## Prerequisites

Before running this notebook:
- ✅ Complete `01_train_model.ipynb` (generates checkpoints)

## Configuration Loading

This section loads the model configuration from a TOML file. The configuration contains all relevant parameters for inference — including model architecture, data sampling intervals, and runtime behavior. It should match the configuration used during training.
 
It is parsed into a `Configuration` object for convenient access throughout the notebook.  

The configuration parameters are also used to compute:
- The number of **context steps** (used for conditioning).
- The number of **prediction steps** (the forecast horizon)


In [ ]:
import pathlib

import tomlkit

from s4casting.core.config import Configuration

with pathlib.Path("../configs/cuda-tiny.toml").open("r", encoding="utf-8") as f:
    config_data = tomlkit.load(f)

config_data["io"]["features"]["measurements"]["location"] = "data/notebook_test_data/external_data_wrapped.json"

config = Configuration(**config_data.unwrap())

config.model.loss.loss = "nll"
config.model.output_head.arch = "gmm"
config.model.ssm.n_layers = 3
config.model.latent_dim = 64
config.model.output_head.n_gaussians = 2

predict_width_days = config.model.predict_width_days  # 2
interval_min = config.model.base_sample_interval_minutes  # 15
context_steps_cfg = int(config.model.input_width_days * 24 * 60 / interval_min)
predict_steps_cfg = int(config.model.predict_width_days * 24 * 60 / interval_min)

print(
    "Context steps:",
    context_steps_cfg,
    "Predict steps:",
    predict_steps_cfg,
    "Total steps:",
    context_steps_cfg + predict_steps_cfg,
    "Patch size:",
    config.model.patch_encoder.patch_size,
    "Interval (min):",
    interval_min,
)

## Loading Input Data

Requirements for the input DataFrame:
- A monotonic datetime index (or a time column set as index).
- A target column (e.g. measurements, load).
- The correct sampling rate the model was trained on, in this example 15 minutes.

For synthetic examples we build the same sort of sinusoid, for real use cases load parquet / csv and align column names.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from tests.utils import create_sinusoid_dataframe

df = create_sinusoid_dataframe(n=864, interval_min=15, start_time="2023-01-01 00:00:00").set_index("timestamp")

df

In [ ]:
import matplotlib.pyplot as plt

series = df["measurements"]

fig, ax = plt.subplots(figsize=(10, 4))
series.plot(ax=ax)
ax.set_xlabel("Time")
ax.set_ylabel("Load")
ax.set_title("Load over Time")
plt.tight_layout()
plt.show()

## Running Inference

The DataFrameInferenceRunner handles:
- Loading the checkpoint weights.
- Creating the context window.
- Forward pass to produce forecast values.
- Storing tensors for plotting and saving.

Important parameters:
- checkpoint_path: which trained state to load.
- target_col: column to extract from the DataFrame.
- save_path: directory where artefacts (pickle, plots) will be written.

With the previous two runs in `01_train_model` we can compare early vs final checkpoint behavior.

- First, we run inference with the first checkpoint.  
- Then, we run inference with the final checkpoint.


In [ ]:
from s4casting.inference.runner import DataFrameInferenceRunner

runner = DataFrameInferenceRunner(
    config=config,
    checkpoint_path="out/checkpoint_1.pt",
    target_col="measurements",
    save_path="out/results",
)
runner.inference(df)

runner.save_predictions_pickle()

fig = runner.plot_results()
fig.show()

In [ ]:
from s4casting.inference.runner import DataFrameInferenceRunner

runner = DataFrameInferenceRunner(
    config=config,
    checkpoint_path="out/last_checkpoint.pt",
    target_col="measurements",
    save_path="out/results",
)
runner.inference(df)

runner.save_predictions_pickle()

fig = runner.plot_results()
fig.show()

### Inspecting Results
In the first run we can see that the model is not yet able to capture the underlying pattern and produces an output only in the right range. In the second run, the model has learned to capture the sinusoidal pattern and produces a forecast that closely follows the sinusoid.

## Summary & Next Steps

You've successfully run inference with a basic S4Casting model!

### Next Steps
1. Explore the generated plots and forecast results in `03_inference.ipynb` with the Ground truth data.
2. Use the inference pipeline on your own model and datasets formatted with `DatasetFormatter`.